In [10]:
# import sys
# sys.path.append("/Users/jong/Documents/ovgu/spine/spine_interaction/src/")

import lightning as pl
import torch
import numpy as np
import pandas as pd

from gait_ml.model.litmodel import LitSeq2Seq
from gait_ml.plotting import plot_xyz

from gait_ml.data.dataset import GaitDataset
from gait_ml.data.datamodule import GaitDataModule
from glob import glob

from scipy.signal import find_peaks, peak_widths, butter, sosfiltfilt

import plotly.io as pio
pio.renderers.default = 'notebook'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1. Visualize Dataset

In [12]:
all_files = glob(
    # "/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_*_T1/*_1_2mW_IPhone.xls",
    "/home/qivy00li/projects/gait_ml/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_*_T1/*_1_2mW_IPhone.xls",
    recursive=True,
)
all_files = np.sort(all_files)
print(f"Processing: {len(all_files)} samples")
window_size = 256
step_size = 64
batch_size = 1
expand_labels=1

# Initialize the GaitDataModule
data_module = GaitDataModule(all_files,
                             batch_size=batch_size,
                             window_size=window_size,
                             step_size=step_size,
                             train_idx=[0,1,3], 
                             val_idx=[2],
                             test_idx=[2],
                             expand_labels=expand_labels,
                             acc_sheet_name="Linear Accelerometer"
                            )

data_module.setup(stage="train")

# # # # Access the dataloaders
# train_loader = data_module.train_dataloader()
# val_loader = data_module.val_dataloader()

# # Plot raw data
# for i in train_loader:
#     print(i[0].shape, i[1].shape)
#     fig = plot_xyz(i[0][0, :, 0], i[0][0, :, 1], i[0][0, :, 2], labels=i[1][0, :])
#     fig.show()
#     fig = plot_xyz(i[0][0, :, 3], i[0][0, :, 4], i[0][0, :, 5], labels=i[1][0, :])
#     fig.show()
#     break

Processing: 46 samples
/home/qivy00li/projects/gait_ml/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_01_T1/01_1_2mW_IPhone.xls
cropped_array shape: (210, 256, 7)
/home/qivy00li/projects/gait_ml/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_02_T1/02_1_2mW_IPhone.xls
cropped_array shape: (196, 256, 7)
/home/qivy00li/projects/gait_ml/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_04_T1/04_1_2mW_IPhone.xls
cropped_array shape: (196, 256, 7)
all_arrays shape: (602, 256, 7)


RuntimeError: Failed to compute zscale stats from train dataset.

### 2 Load data

In [3]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
import wandb

import torch
import numpy as np
import pandas as pd
import lightning as L
from torch.utils.data import TensorDataset, DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from torchmetrics.classification import Accuracy

from gait_ml import evaluate
from glob import glob
from datetime import datetime

In [6]:
all_files = glob(
    "/home/geromevivar/projects/spine_interaction/data/dataset2/10092025/imu/Mobilephone/Termin_1_Vicon/ID_*_T1/*_1_2mW_IPhone.xls",
    recursive=True,
)
all_files = np.sort(all_files)
print(f"Processing: {len(all_files)} samples")

ids = [int(i.split("/")[-1].split("_")[0]) for i in all_files]
window_size = 256
step_size = 128
batch_size = 256
expand_labels = 0
# RNN_TYPE = 

SyntaxError: invalid syntax (1553008491.py, line 13)

In [ ]:
group_df = pd.read_csv("/home/geromevivar/projects/gait_ml/data/processed/groups.csv", index_col="ID")
group_df.columns = ["group", "status"]
group_df = group_df[group_df.group.notna()]
# group_df.drop(index=[6, 22], inplace=True)

# Healthy group = 0 | Pain group = 1
group_df.replace("h", 0, inplace=True)
group_df.replace("p", 1, inplace=True)
grouping = group_df.loc[ids].group.values
grouping

### 3. Train model

In [ ]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store scores
validation_scores = []
test_scores = []

X = np.arange(len(ids)).reshape(-1, 1)
y = grouping

# Outer loop for K-Fold cross-validation
# This loop creates the primary TEST set for each fold.
for fold, (train_val_index, test_index) in enumerate(skf.split(X, y)):
    print(f"=============== FOLD {fold + 1}/{n_splits} ================")

    # Split data into a temporary training+validation set and the final test set
    X_train_val, X_test = X[train_val_index], X[test_index]
    y_train_val, y_test = y[train_val_index], y[test_index]
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=1
    )
    print("X_train: ", X_train.squeeze())
    print("X_val: ", X_val.squeeze())
    print("test_index: ", test_index)

    data_module = GaitDataModule(all_files,
                                 batch_size=batch_size,
                                 window_size=window_size,
                                 step_size=step_size,
                                 train_idx=X_train.squeeze(), 
                                 val_idx=X_val.squeeze(),
                                 test_idx=test_index,
                                 expand_labels=expand_labels,
                                 acc_sheet_name="Linear Accelerometer",
                                 num_workers=16,
                            )

    # Define model parameters
    INPUT_DIM = 6        # As specified by the user
    OUTPUT_DIM = 3       # Changed to 1 as requested
    HIDDEN_DIM = 64
    NUM_LAYERS = 2
    DROPOUT_PROB = 0.5
    LEARNING_RATE = 0.001
    TEACHER_FORCING_RATIO = 0.5
    NUM_EPOCHS = 250

    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    run_name = f"GRU-expandlabel{expand_labels}_{current_time}"
    project_name = "backpain"
    print(f"Starting run: {run_name} in {project_name}")
    
    # Initialize the Lightning Module
    model = LitSeq2Seq(
        input_dim=INPUT_DIM,
        output_dim=OUTPUT_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
        dropout_prob=DROPOUT_PROB,
        learning_rate=LEARNING_RATE,
        teacher_forcing_ratio=TEACHER_FORCING_RATIO,
        
    )
    
    wandb_logger = WandbLogger(
        project=project_name,
        name=run_name,
        log_model="all",
    )
    checkpoint_callback = ModelCheckpoint(
        monitor='val_f1score',          # Metric to monitor
        mode='max',                  # 'min' for loss, 'max' for accuracy
        save_top_k=3,                # Save the top 3 models
        dirpath=f'{project_name}/{run_name}/checkpoints/',      # Directory to save checkpoints
        filename='model-{epoch:02d}-{val_f1score:.2f}' # Checkpoint file name
        
    )
    lr_monitor = LearningRateMonitor(logging_interval='step')

    trainer = pl.Trainer(
        max_epochs=NUM_EPOCHS,
        accelerator="gpu",
        devices=1,      
        log_every_n_steps=1,
        check_val_every_n_epoch=1,
        enable_progress_bar=True,
        logger=wandb_logger,
        callbacks=[checkpoint_callback, lr_monitor]
        # logger=pl.pytorch.loggers.TensorBoardLogger("tb_logs", name="retrain_seq2seq_model") # Uncomment for TensorBoard logging
    )

    print("Starting training...")
    # Train the model
    data_module.setup("train")
    # trainer.fit(model, train_dataloaders=data_module.train_dataloader(), val_dataloaders=data_module.val_dataloader())
    
    print("\nTraining complete!")
    wandb.finish()

    break
    # # 5. Train the model on the TRAIN set
    # model = LogisticRegression(solver='liblinear')
    # model.fit(X_train, y_train)

    # # 6. Evaluate on VALIDATION and TEST sets
    # # The validation set would typically be used for hyperparameter tuning.
    # val_preds = model.predict(X_val)
    # val_accuracy = accuracy_score(y_val, val_preds)
    # validation_scores.append(val_accuracy)
    # print(f"Accuracy on VALIDATION set: {val_accuracy:.4f}")
    
    # # The test set is used for the final performance metric of this fold.
    # test_preds = model.predict(X_test)
    # test_accuracy = accuracy_score(y_test, test_preds)
    # test_scores.append(test_accuracy)
    # print(f"Accuracy on TEST set:       {test_accuracy:.4f}\n")


# 7. Final summary of performance across all folds
# print("=================== SUMMARY ===================")
# print(f"Average Validation Accuracy: {np.mean(validation_scores):.4f} (+/- {np.std(validation_scores):.4f})")
# print(f"Average Test Accuracy:       {np.mean(test_scores):.4f} (+/- {np.std(test_scores):.4f})")

### 4. Eval

In [7]:
from torchmetrics.classification import (
    MulticlassPrecision,
    MulticlassRecall,
    BinaryPrecision,
    BinaryRecall,
    Accuracy,
    ConfusionMatrix,
    MulticlassConfusionMatrix,
    MulticlassF1Score,
    F1Score
)
import matplotlib.pyplot as plt

In [8]:
print(f"Best model: {checkpoint_callback.best_model_path}")
device_ = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
best_model = LitSeq2Seq.load_from_checkpoint(checkpoint_callback.best_model_path).to(device_)

NameError: name 'checkpoint_callback' is not defined

In [9]:
for curr_test_idx in test_index.tolist():
    curr_test_idx=7
    test_datamodule = GaitDataModule(all_files,
                                     batch_size=1024,
                                     window_size=window_size,
                                     step_size=step_size,
                                     train_idx=X_train.squeeze(), 
                                     val_idx=X_val.squeeze(),
                                     test_idx=[curr_test_idx],
                                     expand_labels=expand_labels,
                                     acc_sheet_name="Linear Accelerometer",
                                     num_workers=16,
                                    )
    
    test_datamodule.setup("test")
    test_dataloader = test_datamodule.test_dataloader()
    best_model.eval()
    with torch.no_grad():
        pred=[]
        target=[]
        for sample_input, sample_target in test_dataloader:
            sample_input = sample_input.to(device_)
            sample_target = sample_target.to(device_)
            print(sample_input.shape)
            predicted_output = best_model(sample_input, sample_target, teacher_forcing_ratio=0.0)
            pred.append(predicted_output)
            target.append(sample_target)
        
        cur_pred = torch.concat(pred).reshape(-1, 3)
        cur_pred = torch.nn.functional.softmax(cur_pred, dim=-1).argmax(1)
        cur_target = torch.concat(target).reshape(-1)
        reshaped_input = sample_input.reshape(-1, sample_input.shape[-1]).cpu().numpy()
        
        print("Pred", cur_pred.shape, "Target", cur_target.shape)
        # cm = MulticlassConfusionMatrix(num_classes=3, normalize="true").to(device_)
        # cm.update(cur_pred, cur_target)
        # fig_, ax_ = cm.plot()
        # plt.show()
        
        ALIGN_TOLERANCE = 3
        merged_preds = evaluate.merge_clustered_events(cur_pred.cpu().numpy())
        merged_targets = evaluate.merge_clustered_events(cur_target.cpu().numpy())
        aligned_preds = evaluate.align_events(merged_targets, merged_preds, ALIGN_TOLERANCE)
        
        
        cm = MulticlassConfusionMatrix(num_classes=3, normalize="true")
        cm.update(torch.tensor(aligned_preds), torch.tensor(merged_targets))
        fig_, ax_ = cm.plot()
        plt.show()
        
        num_classes = 3
        # aligned_tensor = torch.tensor(aligned_preds)
        # gt_tensor = torch.tensor(sample_target)
        f1_macro = MulticlassF1Score(num_classes=num_classes, average='macro')
        # f1_per_class = MulticlassF1Score(num_classes=num_classes, average='none')
        # precision_macro = MulticlassPrecision(num_classes=num_classes, average='macro')
        # precision_per_class = MulticlassPrecision(num_classes=num_classes, average='none')
        f1_macro_score = f1_macro(torch.tensor(aligned_preds), torch.tensor(merged_targets))
        print(f"Macro F1-Score: {f1_macro_score.item():.4f} ✨")
        # print(f"Per-Class F1-Scores (0, 1, 2): {f1_per_class(aligned_tensor, gt_tensor)}")
        # print(f"Macro Precision: {precision_macro(aligned_tensor, gt_tensor).item():.4f} ✨")
        # print(f"Per-Class Precision (0, 1, 2): {precision_per_class(aligned_tensor, gt_tensor)}")
        
        # slide_end = 1000
        # reshaped_input = reshaped_input[:slide_end]
        # cur_pred = cur_pred[:slide_end]
        # merged_targets = merged_targets.reshape(-1)[:slide_end]
        # aligned_preds = aligned_preds[:slide_end]
        fig = plot_xyz(reshaped_input[:, 0], reshaped_input[:, 1], reshaped_input[:, 2], labels=aligned_preds)
        fig.show()
        fig = plot_xyz(reshaped_input[:, 0], reshaped_input[:, 1], reshaped_input[:, 2], labels=merged_targets)
        fig.show()
    
        # merged_preds = evaluate.merge_clustered_events(cur_pred.cpu().numpy())
        # ALIGN_TOLERANCE = 3
        # aligned_preds = evaluate.align_events(sample_target, merged_preds, ALIGN_TOLERANCE)
        # sample_target = evaluate.merge_clustered_events(sample_target.numpy())
        # # aligned_preds = evaluate.align_events(ground_truth_np, merged_preds, ALIGN_TOLERANCE)
        # # fig = plot_xyz(reshaped_input[:, 0], reshaped_input[:, 1], reshaped_input[:, 2], labels=merged_preds)
        # # fig.show()
        
        # # fig = plot_xyz(reshaped_input[:, 0], reshaped_input[:, 1], reshaped_input[:, 2], labels=aligned_preds)
        # # fig.show()
    
        # # fig = plot_xyz(reshaped_input[:, 0], reshaped_input[:, 1], reshaped_input[:, 2], labels=sample_target)
        # # fig.show()    
        
        # k+=1
    break

NameError: name 'test_index' is not defined

In [ ]:
# k=0
# for sample_input, sample_target in test_dataloader:
#     print(sample_input.shape, sample_target.shape)
#     cur_pred = torch.nn.functional.softmax(pred[k]).reshape(-1, 3).argmax(1)
#     reshaped_input = sample_input.reshape(-1, 6)
#     slide_end = 1000
#     reshaped_input = reshaped_input[:slide_end]
#     cur_pred = cur_pred[:slide_end]
#     sample_target = sample_target.reshape(-1)[:slide_end]
#     fig = plot_xyz(reshaped_input[:, 0], reshaped_input[:, 1], reshaped_input[:, 2], labels=cur_pred)
#     fig.show()

#     # sample_target = evaluate.merge_clustered_events(sample_target.numpy())
#     merged_preds = evaluate.merge_clustered_events(cur_pred.cpu().numpy())
#     # aligned_preds = evaluate.align_events(ground_truth_np, merged_preds, ALIGN_TOLERANCE)
#     fig = plot_xyz(reshaped_input[:, 0], reshaped_input[:, 1], reshaped_input[:, 2], labels=merged_preds)
#     fig.show()
    
#     ALIGN_TOLERANCE = 3
#     aligned_preds = evaluate.align_events(sample_target, merged_preds, ALIGN_TOLERANCE)
#     fig = plot_xyz(reshaped_input[:, 0], reshaped_input[:, 1], reshaped_input[:, 2], labels=aligned_preds)
#     fig.show()

#     # aligned_preds = evaluate.align_events(ground_truth_np, merged_preds, ALIGN_TOLERANCE)
#     fig = plot_xyz(reshaped_input[:, 0], reshaped_input[:, 1], reshaped_input[:, 2], labels=sample_target)
#     fig.show()    
    
#     num_classes = 3
#     aligned_tensor = torch.tensor(aligned_preds)
#     gt_tensor = torch.tensor(sample_target)
#     f1_macro = MulticlassF1Score(num_classes=num_classes, average='macro')
#     f1_per_class = MulticlassF1Score(num_classes=num_classes, average='none')
#     precision_macro = MulticlassPrecision(num_classes=num_classes, average='macro')
#     precision_per_class = MulticlassPrecision(num_classes=num_classes, average='none')
#     print(f"Macro F1-Score: {f1_macro(aligned_tensor, gt_tensor).item():.4f} ✨")
#     print(f"Per-Class F1-Scores (0, 1, 2): {f1_per_class(aligned_tensor, gt_tensor)}")
#     print(f"Macro Precision: {precision_macro(aligned_tensor, gt_tensor).item():.4f} ✨")
#     print(f"Per-Class Precision (0, 1, 2): {precision_per_class(aligned_tensor, gt_tensor)}")
#     k+=1
#     break

### 4.1 Metrics

In [ ]:
# for curr_pred, curr_target in zip(pred, target):
#     reshaped_pred = curr_pred.reshape(-1, 3)
#     reshaped_target = curr_target.reshape(-1)
#     # --- 1. Define Multiclass Data ---
#     # 0=None, 1=Heel Strike, 2=Toe Off
#     ground_truth_np = reshaped_target.numpy()
#     prediction_np   = reshaped_pred.argmax(axis=1).cpu().numpy()
    
#     # --- 2. Merge and Align ---
#     ALIGN_TOLERANCE = 2
    
#     # Step 2a: Merge the clustered predictions
#     merged_preds = evaluate.merge_clustered_events(prediction_np)
    
#     # Step 2b: Align the cleaned predictions to the ground truth
#     aligned_preds = evaluate.align_events(ground_truth_np, merged_preds, ALIGN_TOLERANCE)
#     ground_truth_np = evaluate.merge_clustered_events(ground_truth_np)
    
#     print(f"Ground Truth:        {ground_truth_np}")
#     print(f"Original Prediction:   {prediction_np}")
#     print(f"Merged Prediction:     {merged_preds}")
#     print(f"Final Aligned Pred:    {aligned_preds}\n")
    
#     # --- 3. Calculate Metrics ---
#     gt_tensor = torch.from_numpy(ground_truth_np)
#     aligned_tensor = torch.from_numpy(aligned_preds)
    
#     num_classes = 3
#     f1_macro = MulticlassF1Score(num_classes=num_classes, average='macro')
#     f1_per_class = MulticlassF1Score(num_classes=num_classes, average='none')
#     precision_macro = MulticlassPrecision(num_classes=num_classes, average='macro')
#     precision_per_class = MulticlassPrecision(num_classes=num_classes, average='none')
#     print(f"Macro F1-Score: {f1_macro(aligned_tensor, gt_tensor).item():.4f} ✨")
#     print(f"Per-Class F1-Scores (0, 1, 2): {f1_per_class(aligned_tensor, gt_tensor)}")
#     print(f"Macro Precision: {precision_macro(aligned_tensor, gt_tensor).item():.4f} ✨")
#     print(f"Per-Class Precision (0, 1, 2): {precision_per_class(aligned_tensor, gt_tensor)}")
    
#     cm = MulticlassConfusionMatrix(num_classes=3, normalize="true")
#     cm.update(aligned_tensor, gt_tensor)
#     fig_, ax_ = cm.plot()
#     plt.show()


### 4.3 Temporal Error

In [ ]:
import numpy as np

def calculate_gait_mae(ground_truth: np.ndarray, prediction: np.ndarray, tolerance: int = 20):
    """
    Matches gait events and computes overall and per-class MAE in one pass.

    Args:
        ground_truth: Array of ground truth labels.
        prediction: Array of predicted labels.
        tolerance: Max distance to consider a match.

    Returns:
        A tuple containing:
        - overall_mae (float): The MAE across all matched events.
        - per_class_mae (dict): A dictionary mapping class labels to their MAE.
    """
    gt_indices = np.where(ground_truth > 0)[0]
    available_preds = list(np.where(prediction > 0)[0])
    
    # Store tuples of (label, absolute_error) for each successful match
    errors_by_class = []
    
    # Greedily match each ground truth event to the closest prediction
    for gt_idx in gt_indices:
        gt_label = ground_truth[gt_idx]
        best_dist = float('inf')
        best_match_idx = -1
        
        # Find the closest available prediction of the same class within tolerance
        for pred_idx in available_preds:
            if prediction[pred_idx] == gt_label:
                dist = abs(gt_idx - pred_idx)
                if dist <= tolerance and dist < best_dist:
                    best_dist = dist
                    best_match_idx = pred_idx
        
        # If a match is found, record its error and remove it from the pool
        if best_match_idx != -1:
            error = abs(gt_idx - best_match_idx)
            errors_by_class.append((gt_label, error))
            available_preds.remove(best_match_idx)
            
    if not errors_by_class:
        return 0.0, {}

    # --- Calculate Final Metrics ---
    
    # Overall MAE is the mean of all collected errors
    all_errors = [err for lbl, err in errors_by_class]
    overall_mae = np.mean(all_errors)
    
    # Per-class MAE is calculated by grouping errors by label
    unique_labels = sorted(np.unique([lbl for lbl, err in errors_by_class]))
    per_class_mae = {
        label: np.mean([err for lbl, err in errors_by_class if lbl == label])
        for label in unique_labels
    }
    
    return overall_mae, per_class_mae

# --- Example Usage ---

# # Sample Data: 0=No Event, 1=Heel Strike, 2=Toe Off
# ground_truth = np.array([0,0,1,0,0,0,2,0,0,1,0,0,0,2,0,0,1,0])
# prediction   = np.array([0,1,0,0,0,2,0,0,0,0,1,0,2,0,0,0,0,1])

# Get metrics with a single function call
overall, per_class = calculate_gait_mae(ground_truth_np, aligned_preds)

print(f"--- Overall MAE ---\n{overall:.4f} frames\n")
print("--- Per-Class MAE ---")
for label, mae in per_class.items():
    print(f"Class {label}: {mae:.4f} frames")

In [ ]:
# Get metrics with a single function call
overall, per_class = calculate_gait_mae(gt_tensor, aligned_tensor)

In [ ]:
overall

In [ ]:
per_class